# Hands-On: Reinforcement Learning for a Vision-Language-Action Robot Policy with RLinf on AMD Radeon™ GPUs

In this workshop you will fine-tune a **Vision-Language-Action (VLA)** robot policy with **reinforcement learning (RL)** in a physics simulator, end to end, on **4 AMD Radeon™ PRO W7900D GPUs**.

| Component | What we use |
| --- | --- |
| RL framework | [RLinf](https://github.com/RLinf/RLinf), a scalable RL infrastructure for embodied and agentic AI |
| Policy | **Lingbot-VLA** (4.2 B parameters), already supervised-fine-tuned (SFT) on RoboTwin tasks |
| Simulator | [RoboTwin 2.0](https://github.com/RoboTwin-Platform/RoboTwin) (SAPIEN physics + Vulkan rendering), dual-arm ALOHA robot |
| Task | `place_shoe`: pick up a shoe and place it on the target mat |
| Algorithm | **GRPO** (Group Relative Policy Optimization) with a task-success reward |
| Hardware / software | 4 × Radeon PRO W7900D (gfx1100, 48 GB), ROCm 7.2, PyTorch 2.11 |

**What you will do in the next ~40 minutes**

1. Check the GPUs and the software environment in your container (~3 min).
2. Walk through the key training parameters (~5 min).
3. Launch a **40-step GRPO training run** that evaluates the policy on held-out scenes **every 10 steps**. The first evaluation result arrives **about 18 minutes** after launch (~55–75 s per training step), then a new one about every 12 minutes, so you will see two to three evaluations during the workshop. The whole run takes about 55 minutes.
4. Watch training progress live, read the metrics, and play the evaluation videos of each evaluation.

> This notebook is a *demo-sized* run: we shrink the number of parallel environments so that every 10 RL steps, including an evaluation, take less than 20 minutes.

## Background: RL Fine-Tuning of a VLA Policy

A VLA policy maps **camera images + robot state + a language instruction** to robot **actions**. Lingbot-VLA predicts an *action chunk*: 50 future joint-space actions for the two arms at a time. The simulator executes the chunk, renders new camera images, and the policy predicts the next chunk. A `place_shoe` episode is at most 400 actions, i.e. 8 chunks.

Supervised fine-tuning (SFT) teaches the policy to imitate demonstrations. **RL fine-tuning** then lets the policy practise in the simulator and reinforces the behaviour that actually completes the task.

### GRPO for robot policies

For each initial scene (object poses, lighting, textures) RLinf runs a **group** of $G$ episodes with the current policy and gives each episode a reward $R_i$, here simply whether the task succeeded:

$$R_i = \mathbb{1}[\text{episode } i \text{ succeeded}], \qquad i = 1, \dots, G$$

Instead of training a value network (critic), GRPO uses the group itself as the baseline:

$$\hat A_i = \frac{R_i - \operatorname{mean}(R_1, \dots, R_G)}{\operatorname{std}(R_1, \dots, R_G)}$$

Episodes that succeed where their siblings failed get a positive advantage; the policy is then updated with the clipped policy-gradient objective of PPO over the log-probabilities of the executed actions:

$$\mathcal{L}(\theta) = -\,\mathbb{E}\Big[\min\big(\rho_t \hat A,\ \operatorname{clip}(\rho_t, 1-\epsilon_\text{low}, 1+\epsilon_\text{high})\,\hat A\big)\Big], \qquad \rho_t = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_\text{old}}(a_t \mid s_t)}$$

The recipe uses an asymmetric clip range, $\epsilon_\text{low} = 0.2$ and $\epsilon_\text{high} = 0.28$ (`algorithm.clip_ratio_low` / `clip_ratio_high`).

If every episode in a group succeeds (or every one fails), all advantages are zero and the group teaches nothing. That is why a **strong SFT policy** (success rate ~90 %) still benefits from RL: the remaining failures are exactly what GRPO learns from.

In this workshop only the **action expert** of Lingbot-VLA (≈0.44 B of the 4.2 B parameters) is trained; the vision-language backbone stays frozen.

### How RLinf runs one training step

RLinf starts three groups of workers with Ray and places all of them on the same 4 GPUs (one worker of each kind per GPU):

```text
                 ┌─────────────────────────── each of the 4 GPUs ───────────────────────────┐
                 │                                                                          │
  RoboTwin <---> │  EnvWorker                MultiStepRolloutWorker      EmbodiedFSDPActor  │
  scenes         │  (SAPIEN: CPU physics,    (policy inference:          (FSDP training     │
                 │   GPU Vulkan rendering)    action chunks + log-probs)  of the policy)    │
                 └──────────────────────────────────────────────────────────────────────────┘

  every training step:
    1. rollout    EnvWorker ⇄ RolloutWorker: every environment plays one episode, in 8 rounds of
                  "observe → predict a chunk of 50 actions → simulate them"
    2. advantage  group-relative advantages from the success rewards
    3. update     Actor: policy-gradient update on the collected trajectories
    4. sync       Actor → RolloutWorker: send the updated weights for the next rollout
  every 10 steps, after the update:
    5. eval       run 32 held-out scenes with the current policy (no training)
```

Workers that are idle in a phase free the GPU: the actor and the rollout worker move their copy of the policy to CPU memory (**offload**), and the env worker closes its simulators and rebuilds them for the next rollout, so all three fit on each 48 GB GPU.

## 1. Verify the AMD GPUs and the Software Environment

### Check the GPUs

`amd-smi` is AMD's command-line tool for monitoring and managing GPUs, the counterpart of `nvidia-smi`. Run without arguments, it shows every GPU visible inside this container: memory used / total (`Mem-Usage`), utilisation (`GFX-Uti`), temperature, power, and the processes that use the GPUs. `amd-smi monitor` prints one compact line per GPU instead.

In [ ]:
!amd-smi

### GPUs and paths

The next cell sets up the run and checks the environment. You do not need to change anything.

- **GPUs**: this instance's 4 GPUs (`GPU_GROUP = "0-3"`).
- **Paths**: the code and Python environment in the image (`/opt/...`), the models and scene assets on the read-only data volume (`/models/rlinf-workshop`), and the output folder of your run (`/workspace/rlinf_workshop`).

Expected output: `OK` on every line, the 4 GPUs at about 0 %, and `All paths found.` If you see `MISSING` or `<-- busy?`, ask an instructor.

In [ ]:
import os
import subprocess
from pathlib import Path

# ---- the 4 GPUs to train on ----
GPU_GROUP = "0-3"

# ---- paths inside the workshop container (prepared by the organizers) ----
RLINF_ROOT    = "/opt/RLinf"                           # RLinf source tree
VENV          = "/opt/venv/lingbotvla"                 # Python environment for Lingbot-VLA
ROBOTWIN_PATH = "/opt/robotwin"                        # RoboTwin checkout; assets in $ROBOTWIN_PATH/assets

def find_model(name):
    # the models are on the read-only data volume at /models/rlinf-workshop (also found under /models or /data/models)
    for pattern in (f"/models/rlinf-workshop/{name}", f"/models/{name}", f"/models/*/{name}",
                    f"/data/models/{name}", f"/data/models/*/{name}"):
        found = sorted(Path("/").glob(pattern.lstrip("/")))
        if found:
            return str(found[0])
    return f"/models/rlinf-workshop/{name}"

POLICY_CKPT   = find_model("lingbot-vla-4b-posttrain-robotwin")   # SFT policy for RoboTwin (weights)
TOKENIZER     = find_model("Qwen2.5-VL-3B-Instruct")              # tokenizer / processor
LINGBOT_CFG   = find_model("lingbot-vla-4b")                      # Lingbot-VLA model config
WORK_DIR      = "/workspace/rlinf_workshop"            # logs, metrics and videos of your runs

RUN_NAME = f"lingbot_place_shoe_grpo_gpu{GPU_GROUP.replace('-', '_')}"
RUN_DIR  = f"{WORK_DIR}/{RUN_NAME}"

# export everything so the %%bash cells below can use it
for k, v in dict(GPU_GROUP=GPU_GROUP, RLINF_ROOT=RLINF_ROOT, VENV=VENV, ROBOTWIN_PATH=ROBOTWIN_PATH,
                 POLICY_CKPT=POLICY_CKPT, TOKENIZER=TOKENIZER, LINGBOT_CFG=LINGBOT_CFG,
                 WORK_DIR=WORK_DIR, RUN_NAME=RUN_NAME, RUN_DIR=RUN_DIR).items():
    os.environ[k] = v

ok = True
for name, path in [("RLinf", RLINF_ROOT), ("Python env", VENV), ("RoboTwin", ROBOTWIN_PATH),
                   ("RoboTwin assets", f"{ROBOTWIN_PATH}/assets/objects"), ("policy checkpoint", POLICY_CKPT),
                   ("tokenizer", TOKENIZER), ("Lingbot-VLA config", LINGBOT_CFG)]:
    exists = Path(path).exists()
    ok &= exists
    print(f"{'OK     ' if exists else 'MISSING'} {name:20s} {path}")

first, last = (int(x) for x in GPU_GROUP.split("-"))
gpus = list(range(first, last + 1))
print(f"\nGPU group {GPU_GROUP} -> GPUs {gpus}")
csv = subprocess.run(["rocm-smi", "--showuse", "--showmemuse", "--csv"], capture_output=True, text=True).stdout
for line in csv.splitlines():
    if line.startswith("card") and int(line.split(",")[0][4:]) in gpus:
        card, use, vram = line.split(",")[:3]
        print(f"  {card}: GPU use {use:>3}%  VRAM {vram:>3}%" + ("   <-- busy?" if int(vram) > 5 else ""))
print("\nAll paths found." if ok else "\nSome paths are missing: ask an instructor before continuing.")

### Verify the Python environment

The container ships one Python environment per model under `/opt/venv`. Activate the Lingbot-VLA environment and check that PyTorch sees the AMD GPUs through ROCm/HIP, and that RLinf and the simulator import:

In [ ]:
%%bash
set -e
source "${VENV}/bin/activate"
cd "${RLINF_ROOT}"
export PYTHONPATH="${RLINF_ROOT}:${ROBOTWIN_PATH}"

(python - <<'PY'
import torch
print("PyTorch      :", torch.__version__, "| HIP", torch.version.hip)
print("GPUs visible :", torch.cuda.device_count(), "x", torch.cuda.get_device_name(0))
import sapien, rlinf, os
print("SAPIEN       :", sapien.__version__)
print("RLinf        :", os.path.dirname(rlinf.__file__))
from rlinf.envs.sim.robotwin.robotwin_env import RoboTwinEnv  # noqa: F401
print("RoboTwin env : import OK")
print("flash-attn backend:", "Triton" if os.environ.get("FLASH_ATTENTION_TRITON_AMD_ENABLE") == "TRUE" else "CK")
PY
) 2>&1 | grep -v "(null): No such file or directory"

## 2. The Training Configuration

RLinf configs are Hydra YAML files. We start from the recipe
`examples/embodiment/config/robotwin_place_shoe_grpo_lingbotvla.yaml` and override a few values on the command line. Print the parts of the recipe that matter here:

In [ ]:
%%bash
source "${VENV}/bin/activate"
python - <<'PY'
import os
from omegaconf import OmegaConf
cfg = OmegaConf.load(os.path.join(os.environ["RLINF_ROOT"], "examples/embodiment/config/robotwin_place_shoe_grpo_lingbotvla.yaml"))
show = {
    "cluster.component_placement": cfg.cluster.component_placement,
    "runner": {k: cfg.runner[k] for k in ("max_steps", "val_check_interval", "save_interval")},
    "algorithm": {k: cfg.algorithm[k] for k in ("adv_type", "group_size", "update_epoch", "clip_ratio_low", "clip_ratio_high")},
    "env.train": {k: cfg.env.train[k] for k in ("total_num_envs", "rollout_epoch", "max_episode_steps", "group_size")},
    "env.eval": {k: cfg.env.eval[k] for k in ("total_num_envs", "rollout_epoch", "max_episode_steps")},
    "rollout": {k: cfg.rollout[k] for k in ("backend", "pipeline_stage_num", "enable_offload")},
    "actor": {"micro_batch_size": cfg.actor.micro_batch_size, "global_batch_size": cfg.actor.global_batch_size,
              "optim.lr": cfg.actor.optim.lr, "enable_offload": cfg.actor.enable_offload},
}
print(OmegaConf.to_yaml(OmegaConf.create({k.replace(".", "/"): v for k, v in show.items()})))
PY

### Workshop overrides and why

The full recipe trains on 128 parallel environments with large batches; one step takes several minutes. For the workshop we keep the algorithm and shrink the batch so that every 10 steps, including an evaluation, take less than 20 minutes.

| Override | Recipe default | Workshop | Why |
| --- | --- | --- | --- |
| `cluster.component_placement` | `actor, env, rollout: 0-7` | `{actor: <group>, env: <group>, rollout: <group>}` | Use only your 4 GPUs; one worker of each kind per GPU |
| `env.train.total_num_envs` | 128 | **16** | 4 environments per GPU: step time grows with environments per GPU |
| `algorithm.group_size` | 8 | **4** | GRPO group size; with 16 environments this gives 4 groups per step |
| `env.train.rollout_epoch` | 8 | **1** | One episode per environment per step |
| `actor.global_batch_size` / `actor.micro_batch_size` | 1024 / 32 | **16 / 4** | All 16 trajectories form one global batch; `global_batch_size` must be divisible by `micro_batch_size × 4 GPUs` |
| `algorithm.update_epoch` | 2 | **1** | One pass over each batch |
| `env.eval.total_num_envs` | 128 | **32** | 32 held-out evaluation episodes |
| `runner.max_steps` / `runner.val_check_interval` | -1 / 20 | **40 / 10** | Train 40 steps, evaluate after steps 10, 20, 30 and 40 |
| `runner.save_interval` | 20 | **-1** | No checkpoints (each would be ~18 GB) |
| `env.eval.video_cfg.save_video` / `fps` | True / 30 | **True / 2** | Save evaluation videos; one frame per action chunk, played at 2 fps |
| `env.train.video_cfg.save_video` | True | **False** | Skip training videos |

Constraints RLinf checks at start-up:

- `total_num_envs / num_GPUs` must be divisible by `group_size` (each GPU holds whole groups),
- `global_batch_size` must be divisible by `micro_batch_size × num_GPUs`.

> **On AMD Radeon GPUs, keep `rollout.pipeline_stage_num: 1` (the default) and do not enable `runner.overlap_env_bootstrap`.** Both make Vulkan rendering and HIP compute run on the same GPU at the same time; on Radeon PRO W7900D this has caused GPU hangs (`HW Exception ... GPU Hang`, `vk::Device::waitForFences: ErrorDeviceLost`). They also save little time here: the step time is dominated by resetting the RoboTwin scenes.

## 3. Launch Training

The next cell starts training **in the background** and returns immediately. It runs RLinf's training script for embodied agents with the recipe from section 2 and the workshop overrides from the table above:

```bash
cd /opt/RLinf
python examples/embodiment/train_embodied_agent.py \
    --config-path /opt/RLinf/examples/embodiment/config \
    --config-name robotwin_place_shoe_grpo_lingbotvla \
    <workshop overrides>
```

`train_embodied_agent.py` starts Ray, launches the env, rollout and actor workers on your 4 GPUs and runs the training loop described in "How RLinf runs one training step". Before that, the cell activates the Python environment and adds RLinf and RoboTwin to `PYTHONPATH`.

Logs go to `$RUN_DIR/train.log` and metrics to `$RUN_DIR/metrics.log`. Re-running the cell while a run is active does nothing.

In [ ]:
%%bash
set -euo pipefail
mkdir -p "${RUN_DIR}"
LOG_FILE="${RUN_DIR}/train.log"
PID_FILE="${RUN_DIR}/train.pid"

# a process counts as running unless it is gone or a zombie (finished, not yet reaped)
is_running() { local s; s=$(ps -o stat= -p "$1" 2>/dev/null); [[ -n "${s}" && "${s}" != Z* ]]; }
if [[ -f "${PID_FILE}" ]] && is_running "$(cat "${PID_FILE}")"; then
    echo "Training is already running (PID $(cat "${PID_FILE}")). Log: ${LOG_FILE}"
    exit 0
fi

nohup setsid bash -s >"${LOG_FILE}" 2>&1 <<'TRAIN_SCRIPT' &
set -uo pipefail
source "${VENV}/bin/activate"
cd "${RLINF_ROOT}"
export PYTHONPATH="${RLINF_ROOT}:${ROBOTWIN_PATH}"
export REPO_PATH="${RLINF_ROOT}" EMBODIED_PATH="${RLINF_ROOT}/examples/embodiment"
export ROBOT_PLATFORM=ALOHA ROBOTWIN_PATH="${ROBOTWIN_PATH}"
export LINGBOT_VLA_PATH="${LINGBOT_VLA_PATH:-${VENV}/lingbot-vla}"
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl TOKENIZERS_PARALLELISM=false PYTHONUNBUFFERED=1 HYDRA_FULL_ERROR=1

echo "TRAIN_START $(date -u '+%F %T') UTC  GPU group ${GPU_GROUP}"
python examples/embodiment/train_embodied_agent.py \
    --config-path "${RLINF_ROOT}/examples/embodiment/config" \
    --config-name robotwin_place_shoe_grpo_lingbotvla \
    '~cluster.component_placement' \
    "+cluster.component_placement={actor:${GPU_GROUP},env:${GPU_GROUP},rollout:${GPU_GROUP}}" \
    runner.logger.log_path="${RUN_DIR}" \
    runner.max_steps=40 runner.val_check_interval=10 runner.save_interval=-1 \
    algorithm.group_size=4 algorithm.update_epoch=1 \
    env.train.total_num_envs=16 env.train.rollout_epoch=1 \
    env.eval.total_num_envs=32 \
    actor.global_batch_size=16 actor.micro_batch_size=4 \
    env.train.assets_path="${ROBOTWIN_PATH}" env.eval.assets_path="${ROBOTWIN_PATH}" \
    env.train.video_cfg.save_video=False \
    env.eval.video_cfg.save_video=True +env.eval.video_cfg.fps=2 \
    actor.model.model_path="${POLICY_CKPT}" \
    actor.model.tokenizer_path="${TOKENIZER}" \
    actor.model.lingbotvla.config_path="${LINGBOT_CFG}"
echo "TRAIN_EXIT=$?  $(date -u '+%F %T') UTC"
TRAIN_SCRIPT

echo $! > "${PID_FILE}"
echo "Training started in the background."
echo "  PID : $(cat "${PID_FILE}")"
echo "  Log : ${LOG_FILE}"

### Watch training live

Run the next cell to follow the training. It refreshes every 15 seconds and shows:

- **Status**: running or finished, time since launch, and steps done out of 40.
- **Phase**: what the run is doing right now: starting up, rollout, policy update or evaluation.
- **Evaluation success** after steps 10, 20, 30 and 40.
- **RLinf's metric tables** from `$RUN_DIR/metrics.log`: one table per finished step, the newest at the bottom (section 4 explains the entries).

The cell **stops by itself when a new evaluation result arrives**. Then watch the new evaluation videos in section 5, and run this cell again to keep watching. You can also stop it any time with **Stop (■)**: this only stops watching, **training keeps running**.

Expected timeline on 4 × W7900D:

| Phase | Time |
| --- | --- |
| Start-up: Ray, loading the models, building the environments | ~2.5 min |
| Step 1 (includes the first inference and weight sync) | ~130 s |
| Each later step | ~55–75 s |
| Each evaluation (32 episodes) | ~3 min the first time, then ~1.5 min; longer if other jobs use the CPUs |
| **First evaluation result (after step 10)** | **~18 min after launch** |
| Next evaluations | every ~12 min |
| Whole 40-step run | ~55 min |

In [ ]:
import calendar, re, time
from IPython.display import clear_output, display

ANSI = re.compile(r"\x1b\[[0-9;]*m")

def parse_metrics(path):
    # Parse RLinf's console metric tables into one dict per step.
    try:
        txt = ANSI.sub("", Path(path).read_text(errors="ignore"))
    except FileNotFoundError:
        return []
    rows = []
    for block in txt.split("Metric Table")[1:]:
        m = re.search(r"Global Step:\s+(\d+)/(\d+)", block)
        if not m or "╰" not in block:  # no bottom border yet: RLinf is still writing this table
            continue
        row, section = {"step": int(m.group(1)), "of": int(m.group(2))}, ""
        for line in block.splitlines():
            h = re.search(r"├─+\s*([A-Za-z/ ]+?)\s*─+┤", line)
            if h:
                section = h.group(1).strip()
                continue
            for k, v in re.findall(r"([A-Za-z_/]+)=([-0-9.e+]+)", line):
                try:
                    row[f"{section}:{k}"] = float(v)
                except ValueError:
                    pass
        rows.append(row)
    return rows

def training_status():
    pid_file, log_file = Path(RUN_DIR, "train.pid"), Path(RUN_DIR, "train.log")
    log = ANSI.sub("", log_file.read_text(errors="ignore")) if log_file.exists() else ""
    m = re.search(r"TRAIN_EXIT=(\d+)", log)
    if m:
        return ("finished" if m.group(1) == "0" else f"FAILED (exit {m.group(1)})"), log
    stat = subprocess.run(["ps", "-o", "stat=", "-p", pid_file.read_text().strip()],
                          capture_output=True, text=True).stdout.strip() if pid_file.exists() else ""
    alive = bool(stat) and not stat.startswith("Z")  # a zombie has finished but was not reaped yet
    return ("running" if alive else "not running"), log

def current_phase(log, rows):
    # Each step ends with a metric table in the log: only look at what came after the last one.
    current = log.split("Metric Table")[-1].splitlines()
    evals = [line for line in current if "Evaluating Rollout Epochs" in line]
    rollouts = [line for line in current if "Generating Rollout Epochs" in line]
    if evals:
        return "evaluation (32 episodes)" if "100%" not in evals[-1] else "evaluation done, finishing the step"
    if rollouts:
        return "rollout (16 environments)" if "100%" not in rollouts[-1] else "policy update (training the action expert)"
    return "next step starting" if rows else "starting: Ray, loading the models, building the environments"

def since_launch(log):
    m = re.search(r"TRAIN_START (\S+ \S+) UTC", log)
    if not m:
        return "-"
    start = calendar.timegm(time.strptime(m.group(1), "%Y-%m-%d %H:%M:%S"))  # the log time is UTC
    return time.strftime("%M:%S" if time.time() - start < 3600 else "%H:%M:%S", time.gmtime(time.time() - start))

def show(rows, status, log):
    clear_output(wait=True)
    done = rows[-1]["step"] if rows else 0
    total = rows[-1]["of"] if rows else 40
    header = f"[{time.strftime('%H:%M:%S')}] status: {status} | since launch {since_launch(log)} | steps done {done}/{total}"
    print(header)
    print(f"phase : {current_phase(log, rows)}")
    evals = [(r["step"], r["Evaluation:success_once"]) for r in rows if "Evaluation:success_once" in r]
    if evals:
        print("evaluation success: " + "   ".join(f"step {s}: {v:.3f}" for s, v in evals))
    metrics_log = Path(RUN_DIR, "metrics.log")
    if not metrics_log.exists():
        print("\nNo metric table yet: RLinf writes one to metrics.log after each step.")
        return
    print(f"\n{metrics_log} (RLinf's metric table of every finished step, oldest first):")
    print(metrics_log.read_text(errors="ignore"))
    print(header)  # repeated here, below the newest table

def eval_steps(rows):
    return [r["step"] for r in rows if "Evaluation:success_once" in r]

seen = eval_steps(parse_metrics(Path(RUN_DIR, "metrics.log")))  # evaluations already shown before this run of the cell
try:
    while True:
        status, log = training_status()
        rows = parse_metrics(Path(RUN_DIR, "metrics.log"))
        show(rows, status, log)
        new = [s for s in eval_steps(rows) if s not in seen]
        if new:
            print(f"\nNew evaluation after step {new[-1]}: watch its videos in section 5, "
                  "then re-run this cell to keep watching.")
            break
        if status != "running":
            break
        time.sleep(15)
except KeyboardInterrupt:
    print("\nStopped watching (training keeps running in the background).")

**The two log files.** Training writes two files to `$RUN_DIR` (`/workspace/rlinf_workshop/lingbot_place_shoe_grpo_gpu0_3`):

- `train.log`: everything the training prints: start-up messages, progress bars, warnings and errors from all workers, and the metric tables. Look here when something goes wrong.
- `metrics.log`: only the metric table of each finished step, written by RLinf. The cell above shows this file.

To follow `train.log` live, open a terminal (**File → New → Terminal**) and run:

```bash
tail -f /workspace/rlinf_workshop/lingbot_place_shoe_grpo_gpu0_3/train.log
```

Press **Ctrl+C** to stop following; training keeps running.

## 4. Understand the Metrics

RLinf prints a metric table after every step (also saved in `metrics.log` and as TensorBoard events under `$RUN_DIR/tensorboard`). The most useful entries:

**Environment and evaluation (task performance)**

| Metric | Meaning |
| --- | --- |
| `Environment:success_once` | Fraction of this step's training episodes in which the task succeeded at least once |
| `Environment:num_trajectories` | Training episodes collected this step (16) |
| `Evaluation:success_once` | Success rate on the held-out evaluation scenes, **the headline number** |
| `Evaluation:success_at_end` | Fraction of evaluation episodes still in the success state at the last step |
| `Evaluation:episode_len` | Episode length in actions (400 = the limit; always 400 here, because every episode is run to the limit) |

**Rollout and training (RL signal)**

| Metric | Meaning |
| --- | --- |
| `Rollout:advantages_mean/min/max` | Group-relative advantages; all zeros means every group was all-success or all-failure |
| `Training/Actor:actor/policy_loss` | Clipped policy-gradient loss |
| `Training/Actor:actor/approx_kl`, `actor/clip_fraction` | How far one update moved the policy; small values mean a stable update |
| `Training/Actor:actor/grad_norm` | Gradient norm before clipping |
| `Training/Actor:actor/lr` | Learning rate |

**Time (seconds per step)**

| Metric | Meaning |
| --- | --- |
| `Time:step` | Whole step |
| `Time:generate_rollouts` | Rollout phase: scene reset (`env/env/bootstrap_step`), simulation and policy inference (`rollout/predict`) |
| `Time:actor_training` | Policy update |
| `Time:sync_weights` | Sending the new weights from the actor to the rollout workers |
| `Time:eval` | Evaluation (only on evaluation steps) |

With a strong SFT policy the success rate starts around 0.9, so these 40 small steps are about seeing the full RL loop work, not about large gains. Note how noisy `Environment:success_once` is with only 16 episodes per step.

## 5. Evaluation Results and Videos

After steps 10, 20, 30 and 40, RLinf evaluates the policy on 32 held-out scenes and writes one video per GPU worker and evaluation to `$RUN_DIR/video/eval/seed_<worker seed>/<n>.mp4`, where `<n>` = 0, 1, 2, 3 is the first, second, ... evaluation. Each video tiles that worker's 8 evaluation environments; the overlay shows each episode's reward and termination flag. One frame is recorded per 50-action chunk, so a 400-action episode is about 10 frames; the very last frame already shows the scenes reset for the next episode.

In [ ]:
from IPython.display import HTML
import base64

EVAL_INDEX = -1   # which evaluation to show: -1 = latest, 0 = after step 10, 1 = after step 20, ...

rows = parse_metrics(Path(RUN_DIR, "metrics.log"))
evals = [r for r in rows if "Evaluation:success_once" in r]
if not evals:
    print("No evaluation yet: the first one runs after step 10.")
else:
    print(f"{'after step':>10} {'success':>8} {'episodes':>9} {'success_at_end':>15}")
    for r in evals:
        n = int(r.get("Evaluation:num_trajectories", 0))
        print(f"{r['step']:>10} {r['Evaluation:success_once']:>8.3f} {round(r['Evaluation:success_once'] * n):>4}/{n:<4} "
              f"{r.get('Evaluation:success_at_end', 0):>15.3f}")
    idx = EVAL_INDEX % len(evals)
    videos = sorted(Path(RUN_DIR, "video", "eval").glob(f"seed_*/{idx}.mp4"))
    print(f"\nVideos of the evaluation after step {evals[idx]['step']}: {len(videos)} file(s)")
    display(HTML("".join(
        f"<div style='display:inline-block;margin:4px;text-align:center'><video width='420' controls loop autoplay muted "
        f"src='data:video/mp4;base64,{base64.b64encode(v.read_bytes()).decode()}'></video><br>{v.parent.name}/{v.name}</div>"
        for v in videos)))

## 6. Stop Training and Clean Up

The run stops by itself after the evaluation at step 40 (about 55 minutes after launch). **If it is still running at the end of the workshop, run this cell to free the GPUs.** You can also use it to stop a run early. It interrupts the training process group, waits for Ray to shut down, and then stops any Ray processes left in this container.

In [ ]:
%%bash
PID_FILE="${RUN_DIR}/train.pid"
is_running() { local s; s=$(ps -o stat= -p "$1" 2>/dev/null); [[ -n "${s}" && "${s}" != Z* ]]; }
if [[ -f "${PID_FILE}" ]] && is_running "$(cat "${PID_FILE}")"; then
    PID=$(cat "${PID_FILE}")
    echo "Stopping training (process group ${PID}) ..."
    kill -INT -- "-${PID}" 2>/dev/null || true
    for i in $(seq 1 60); do is_running "${PID}" || break; sleep 1; done
    if is_running "${PID}"; then echo "Still running, sending SIGTERM"; kill -TERM -- "-${PID}" 2>/dev/null || true; sleep 10; fi
    source "${VENV}/bin/activate" && ray stop >/dev/null 2>&1 || true
    echo "Stopped."
else
    echo "No training process is running."
fi
first=${GPU_GROUP%-*}; last=${GPU_GROUP#*-}
echo "GPU group ${GPU_GROUP}:"
rocm-smi --showuse --showmemuse --csv | awk -F, -v a="${first}" -v b="${last}" \
    '/^card/ {n = substr($1, 5) + 0; if (n >= a && n <= b) printf "  %s: GPU use %s%%  VRAM %s%%\n", $1, $2, $3}'

## Troubleshooting

**Harmless log lines on AMD GPUs**

| Log line | Why it is harmless |
| --- | --- |
| `[svulkan2] [error] CUDA Error: cudaErrorInsufficientDriver` / `Failed to initialize denoiser` | SAPIEN probes for an NVIDIA denoiser; rendering uses Vulkan and works without it |
| `ModuleNotFoundError: No module named 'curobo'` (from `planner.py`) | The CUDA-only cuRobo planner is not installed; RoboTwin uses the `mplib` planner |
| `Objects is unstable in seed(...), trying new seed` | RoboTwin rejects a randomly generated scene and samples another one |
| `'MessageFactory' object has no attribute 'GetPrototype'` | protobuf version mismatch reported when a dependency imports TensorFlow; no effect |

**Problems**

| Symptom | Fix |
| --- | --- |
| A GPU in your group shows memory in use before you start | Another process is using it: choose other GPUs or ask an instructor |
| `AssertionError: ... must be divisible by ...` at start-up | See the two divisibility rules in section 2 |
| The run fails with `HW Exception ... GPU Hang` or `ErrorDeviceLost` | Make sure `rollout.pipeline_stage_num` is 1 and `runner.overlap_env_bootstrap` is not set; then relaunch |
| Host out of memory | One run peaks at ~170 GB of host RAM; do not start a second run in the same container |